# 07 — Final training comparison and follow-up evaluation

Run this notebook from a **fresh Colab T4 GPU runtime**, from top to bottom. It is independent of notebook 05 variables. Existing notebooks, runs and selected-model packages are preserved.

1. Setup and data cells verify the existing grouped splits and load training/validation images.
2. The training cell runs one bounded 224px ResNet50 focal-loss experiment (Experiment 6).
3. The comparison cell compares all recorded trained checkpoints using validation macro F1 and exports a verified candidate. It can retain an earlier run if it scores higher.
4. Review the candidate, then explicitly enable the final evaluation cell. The existing test set has already been inspected: these are **follow-up** results, not an untouched independent final estimate.

**No model is declared ready based on validation alone.** Current darker-skin representation is inadequate: the IV–VI validation group has 3 BCC and 1 SCC images, with no Melanoma. Focal loss cannot supply missing patient diversity. Model integration remains paused until the requirements and intended use are reviewed.

Runtime expectation: TensorFlow/Keras with CategoricalFocalCrossentropy (the prior runs used TensorFlow 2.21 / Keras 3.13). Do not install the local Streamlit requirements into this Colab runtime. Class order: BCC=0, Melanoma=1, SCC=2. Loss alpha uses training-derived balance weights; no additional class_weight or oversampling.


## Environment, split checks and Experiment 6 settings


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import gc
import hashlib
import json
import shutil
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display

print('TensorFlow:', tf.__version__)
print('Keras:', getattr(tf.keras, '__version__', 'bundled with TensorFlow'))
gpus = tf.config.list_physical_devices('GPU')
print('Available GPUs:', gpus)
if not gpus:
    raise RuntimeError('Enable the T4 GPU before continuing.')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        print('GPU already initialized; keeping its memory configuration.')

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
HEAD_EPOCHS = 6
FINE_TUNE_EPOCHS = 8
CLASS_NAMES = ['BCC', 'Melanoma', 'SCC']
CLASS_TO_INDEX = {name: index for index, name in enumerate(CLASS_NAMES)}
PROJECT_ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
SPLIT_ROOT = PROJECT_ROOT / 'Datasets' / 'Splits'
MODEL_ROOT = PROJECT_ROOT / 'Models' / 'Transfer_Learning'
REPORT_ROOT = PROJECT_ROOT / 'Reports' / 'Training'
LOCAL_ROOT = Path('/content/skin_cancer_training_images')
for folder in [MODEL_ROOT, REPORT_ROOT, LOCAL_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)

def write_json(path, value):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)
frames = {}
manifest_hashes = {}
for split in ['train', 'validation', 'test']:
    path = SPLIT_ROOT / f'{split}_metadata.csv'
    if not path.is_file():
        raise FileNotFoundError(path)
    manifest_hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()
    frame = pd.read_csv(path, dtype={'patient_id': 'string', 'lesion_id': 'string'})
    required = {'dataset', 'image_id', 'class_label', 'image_path', 'image_hash', 'group_id', 'label_index', 'split'}
    if not required.issubset(frame.columns):
        raise ValueError(f'{split}: required split columns are missing.')
    if not frame['split'].eq(split).all():
        raise ValueError(f'{split}: incorrect split assignments.')
    if set(frame['class_label']) != set(CLASS_NAMES):
        raise ValueError(f'{split}: expected all three classes.')
    expected_labels = frame['class_label'].map(CLASS_TO_INDEX)
    if not expected_labels.eq(frame['label_index']).all():
        raise ValueError(f'{split}: class order differs from notebook 4.')
    frames[split] = frame

combined = pd.concat(frames.values(), ignore_index=True)
assert len(combined) == 2703
assert not combined.duplicated(['dataset', 'image_id']).any()
assert combined.groupby('group_id')['split'].nunique().max() == 1
assert combined.groupby('image_hash')['split'].nunique().max() == 1
pad = combined[combined['dataset'] == 'PAD-UFES-20']
assert pad['patient_id'].notna().all()
assert pad.groupby('patient_id')['split'].nunique().max() == 1
lesions = combined.dropna(subset=['lesion_id'])
assert lesions.groupby(['dataset', 'lesion_id'])['split'].nunique().max() == 1

train_frame = frames['train'].copy()
val_frame = frames['validation'].copy()
train_counts = train_frame['label_index'].value_counts().reindex(range(3))
CLASS_WEIGHTS = {
    index: float(len(train_frame) / (3 * train_counts[index]))
    for index in range(3)
}
print('Training records:', len(train_frame))
print('Validation records:', len(val_frame))
print('Test records reserved for notebook 6:', len(frames['test']))
print('Class order:', CLASS_TO_INDEX)
print('Training-only class weights:', CLASS_WEIGHTS)
display(pd.crosstab(combined['split'], combined['class_label']))


if not hasattr(tf.keras.losses, 'CategoricalFocalCrossentropy'):
    raise RuntimeError('This runtime needs a TensorFlow/Keras version with CategoricalFocalCrossentropy. Do not downgrade to the old app requirements.')

baseline_folder = MODEL_ROOT / 'exp02_65b8c20800f3' / 'ResNet50'
baseline = json.loads((baseline_folder / 'metadata.json').read_text())
if not baseline.get('training_complete') or baseline['manifest_hashes'] != manifest_hashes:
    raise ValueError('Baseline incomplete or split manifests changed.')

HEAD_EPOCHS = 12
FINE_TUNE_EPOCHS = 25
FOCAL_ALPHA = [CLASS_WEIGHTS[index] for index in range(3)]
EXPERIMENT_SETTINGS = {
    'experiment': 'experiment_06_focal',
    'model_name': 'ResNet50',
    'baseline_run_id': baseline['run_id'],
    'seed': SEED, 'image_size': list(IMAGE_SIZE),
    'batch_size': BATCH_SIZE, 'head_epochs': HEAD_EPOCHS,
    'fine_tune_epochs': FINE_TUNE_EPOCHS,
    'head_learning_rate': 0.001, 'fine_tune_learning_rate': 0.00001,
    'dense_units': 128, 'dropout': 0.30, 'fine_tune_fraction': 0.50,
    'early_stopping_patience': 6, 'reduce_lr_patience': 3,
    'loss': 'CategoricalFocalCrossentropy', 'gamma': 2.0,
    'alpha': FOCAL_ALPHA,
    'sampling': 'natural training distribution; no oversampling',
    'fit_class_weight': None,
    'augmentation': 'flip_horizontal_vertical; rotation_0.05; zoom_0.10; brightness_0.10',
    'class_names': CLASS_NAMES, 'manifest_hashes': manifest_hashes,
    'tensorflow_version': tf.__version__,
    'original_test_results_already_inspected': True,
}
RUN_ID = 'exp06_' + hashlib.sha256(json.dumps(EXPERIMENT_SETTINGS, sort_keys=True).encode()).hexdigest()[:12]
RUN_MODELS = MODEL_ROOT / RUN_ID
RUN_REPORTS = REPORT_ROOT / RUN_ID
RUN_MODELS.mkdir(parents=True, exist_ok=True)
RUN_REPORTS.mkdir(parents=True, exist_ok=True)
write_json(RUN_REPORTS / 'training_settings.json', EXPERIMENT_SETTINGS)
print('EXPERIMENT 6 READY:', RUN_ID)
print('Focal alpha (training-only weights):', FOCAL_ALPHA)
print('No extra class_weight or oversampling will be applied.')


## Prepare local training/validation images


In [ ]:
# Copy only training and validation images; test images are not loaded.
for split, frame in [('train', train_frame), ('validation', val_frame)]:
    local_paths = []
    for position, row in enumerate(frame.itertuples(index=False)):
        source = Path(row.image_path)
        if not source.is_file():
            raise FileNotFoundError(source)
        if source.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.bmp'}:
            raise ValueError(f'Unsupported training image format: {source}')
        key = hashlib.sha256(str(source).encode()).hexdigest()
        destination = LOCAL_ROOT / (key + source.suffix.lower())
        if not destination.is_file() or destination.stat().st_size != source.stat().st_size:
            temporary = destination.with_suffix(destination.suffix + '.tmp')
            shutil.copyfile(source, temporary)
            temporary.replace(destination)
        local_paths.append(str(destination))
        if (position + 1) % 250 == 0:
            print(f'{split}: {position + 1}/{len(frame)} copied')
    frame['local_path'] = local_paths
    print(f'{split}: local images ready')

def decode_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(tf.cast(image, tf.float32), IMAGE_SIZE, method='bilinear')
    return image, label

def raw_dataset(frame):
    ds = tf.data.Dataset.from_tensor_slices((
        frame['local_path'].to_numpy(dtype=str),
        frame['label_index'].to_numpy(dtype=np.int32),
    ))
    return ds.map(decode_image, num_parallel_calls=tf.data.AUTOTUNE).cache()

RAW_TRAIN = raw_dataset(train_frame)
RAW_VAL = raw_dataset(val_frame)
# Fully populate caches once, before model training.
for _ in RAW_TRAIN.batch(BATCH_SIZE):
    pass
for _ in RAW_VAL.batch(BATCH_SIZE):
    pass
VAL_LABELS = val_frame['label_index'].to_numpy(dtype=np.int32)
images, labels = next(iter(RAW_TRAIN.batch(6)))
fig, axes = plt.subplots(2, 3, figsize=(10, 7))
for axis, image, label in zip(axes.flat, images.numpy(), labels.numpy()):
    axis.imshow(np.clip(image, 0, 255).astype('uint8'))
    axis.set_title(CLASS_NAMES[int(label)])
    axis.axis('off')
plt.tight_layout()
plt.show()


## Focal-loss pipeline and checkpoint helpers


In [ ]:
MODEL_SPECS = {
    'MobileNetV2': (tf.keras.applications.MobileNetV2, tf.keras.applications.mobilenet_v2.preprocess_input, 'mobilenet_v2'),
    'EfficientNetB0': (tf.keras.applications.EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input, 'efficientnet_b0'),
    'ResNet50': (tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input, 'resnet50'),
}

def model_datasets(name):
    preprocess = MODEL_SPECS[name][1]
    augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal_and_vertical', seed=SEED),
        tf.keras.layers.RandomRotation(0.05, fill_mode='reflect', seed=SEED + 1),
        tf.keras.layers.RandomZoom(0.10, fill_mode='reflect', seed=SEED + 2),
        tf.keras.layers.RandomBrightness(0.10, value_range=(0, 255), seed=SEED + 3),
    ])
    def prepare_training(images, labels):
        images = augmentation(images, training=True)
        return preprocess(tf.clip_by_value(images, 0.0, 255.0)), tf.one_hot(labels, depth=3)
    def prepare_validation(images, labels):
        return preprocess(images), tf.one_hot(labels, depth=3)
    train_ds = RAW_TRAIN.shuffle(len(train_frame), seed=SEED, reshuffle_each_iteration=True)
    train_ds = train_ds.batch(BATCH_SIZE).map(prepare_training, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    val_ds = RAW_VAL.batch(BATCH_SIZE).map(prepare_validation, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    return train_ds, val_ds

def classification_scores(labels, probabilities):
    predicted = probabilities.argmax(axis=1)
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (labels, predicted), 1)
    true_positive = np.diag(matrix).astype(float)
    precision = np.divide(true_positive, matrix.sum(axis=0), out=np.zeros(3), where=matrix.sum(axis=0) != 0)
    recall = np.divide(true_positive, matrix.sum(axis=1), out=np.zeros(3), where=matrix.sum(axis=1) != 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(3), where=(precision + recall) != 0)
    return float(f1.mean()), float((labels == predicted).mean())

def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.CategoricalFocalCrossentropy(alpha=FOCAL_ALPHA, gamma=2.0),
        metrics=[tf.keras.metrics.CategoricalAccuracy(name='accuracy')],
    )

class ValidationCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, name, val_ds, model_dir, report_dir):
        super().__init__()
        self.name = name
        self.val_ds = val_ds
        self.model_dir = model_dir
        self.report_dir = report_dir
        self.best = -1.0
        self.best_accuracy = None
        self.best_phase = None
        self.best_epoch = None
        self.phase = 'head'
        self.rows = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs if logs is not None else {}
        probabilities = self.model.predict(self.val_ds, verbose=0)
        macro_f1, accuracy = classification_scores(VAL_LABELS, probabilities)
        logs['val_macro_f1'] = macro_f1
        self.rows.append({'phase': self.phase, 'epoch': epoch + 1, **{key: float(value) for key, value in logs.items()}})
        pd.DataFrame(self.rows).to_csv(self.report_dir / 'training_history.csv', index=False)
        print(f' — validation macro F1: {macro_f1:.4f}')
        if macro_f1 > self.best:
            self.best = macro_f1
            self.best_accuracy = accuracy
            self.best_phase = self.phase
            self.best_epoch = epoch + 1
            self.model.save(self.model_dir / 'best.keras')
            self.model.save_weights(self.model_dir / 'best.weights.h5')
            write_json(self.model_dir / 'checkpoint.json', {
                'validation_macro_f1': macro_f1, 'validation_accuracy': accuracy,
                'phase': self.phase, 'epoch': epoch + 1, 'class_names': CLASS_NAMES,
            })
            print('Best checkpoint saved to Drive.')


def reduce_lr_callback():
    return tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_macro_f1', mode='max', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1,
    )

def stop_callback():
    return tf.keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max', patience=6, restore_best_weights=False)

def train_one_model(name):
    if name != 'ResNet50':
        raise ValueError('This bounded experiment is for ResNet50 only.')
    model_dir = RUN_MODELS / name
    report_dir = RUN_REPORTS / name
    model_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    metadata_path = model_dir / 'metadata.json'
    if metadata_path.is_file():
        saved = json.loads(metadata_path.read_text())
        if saved.get('training_complete') and saved.get('run_id') == RUN_ID and (model_dir / 'best.keras').is_file():
            if hashlib.sha256((model_dir / 'best.keras').read_bytes()).hexdigest() != saved['model_sha256']:
                raise ValueError('Completed checkpoint checksum differs.')
            print(name, 'already complete for this run; skipping.')
            return saved
    # Never overwrite an incomplete attempt: retain its checkpoints/history.
    if (model_dir / 'best.keras').exists() or (report_dir / 'training_history.csv').exists():
        stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
        model_dir.rename(RUN_MODELS / f'{name}_interrupted_{stamp}')
        report_dir.rename(RUN_REPORTS / f'{name}_interrupted_{stamp}')
        model_dir.mkdir()
        report_dir.mkdir()

    tf.keras.backend.clear_session()
    gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    train_ds, val_ds = model_datasets(name)
    constructor, _, preprocess_name = MODEL_SPECS[name]
    base = constructor(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE, 3))
    base.trainable = False
    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='preprocessed_image')
    features = base(inputs, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D()(features)
    features = tf.keras.layers.Dense(128, activation='relu', name='research_head')(features)
    features = tf.keras.layers.Dropout(0.30, seed=SEED)(features)
    outputs = tf.keras.layers.Dense(3, activation='softmax', name='class_scores')(features)
    model = tf.keras.Model(inputs, outputs, name=name + '_skin_classifier')
    monitor = ValidationCheckpoint(name, val_ds, model_dir, report_dir)
    started = time.time()

    print('\nTRAINING CLASSIFICATION HEAD:', name)
    compile_model(model, 0.001)
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
              callbacks=[monitor, reduce_lr_callback(), stop_callback()], verbose=2)
    model.load_weights(model_dir / 'best.weights.h5')

    print('\nFINE-TUNING:', name)
    base.trainable = True
    cutoff = int(len(base.layers) * 0.50)
    for index, layer in enumerate(base.layers):
        layer.trainable = index >= cutoff and not isinstance(layer, tf.keras.layers.BatchNormalization)
    compile_model(model, 0.00001)
    monitor.phase = 'fine_tune'
    model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS,
              callbacks=[monitor, reduce_lr_callback(), stop_callback()], verbose=2)

    # Validate serialization: loaded best model must reproduce its weights.
    model.load_weights(model_dir / 'best.weights.h5')
    reference_batch, _ = next(iter(val_ds))
    reference_scores = model.predict(reference_batch, verbose=0)
    loaded = tf.keras.models.load_model(model_dir / 'best.keras', compile=False)
    reloaded_scores = loaded.predict(reference_batch, verbose=0)
    np.testing.assert_allclose(reference_scores, reloaded_scores, rtol=1e-4, atol=1e-5)
    np.savez(report_dir / 'validation_reference.npz', scores=reloaded_scores, labels=VAL_LABELS[:len(reloaded_scores)])
    val_frame.head(len(reloaded_scores)).to_csv(report_dir / 'validation_reference_images.csv', index=False)

    # Full validation diagnostics for the saved best checkpoint.
    from sklearn.metrics import classification_report, confusion_matrix
    validation_scores = loaded.predict(val_ds, verbose=0)
    predicted = validation_scores.argmax(axis=1)
    report = classification_report(
        VAL_LABELS, predicted, labels=[0, 1, 2],
        target_names=CLASS_NAMES, output_dict=True, zero_division=0,
    )
    class_report = pd.DataFrame([
        {'class_label': label, **report[label]} for label in CLASS_NAMES
    ])
    class_report.to_csv(report_dir / 'validation_per_class_metrics.csv', index=False)
    display(class_report)
    pd.DataFrame(confusion_matrix(VAL_LABELS, predicted, labels=[0, 1, 2]),
                 index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        report_dir / 'validation_confusion_matrix.csv')
    predictions = val_frame.drop(columns=['local_path'], errors='ignore').copy()
    predictions['predicted_class'] = [CLASS_NAMES[i] for i in predicted]
    for index, label in enumerate(CLASS_NAMES):
        predictions[f'score_{label}'] = validation_scores[:, index]
    predictions.to_csv(report_dir / 'validation_predictions.csv', index=False)
    predictions.loc[predictions['class_label'] != predictions['predicted_class']].to_csv(
        report_dir / 'validation_errors.csv', index=False)
    best_f1, best_accuracy = classification_scores(VAL_LABELS, validation_scores)
    np.testing.assert_allclose(best_f1, monitor.best, atol=1e-8)
    np.testing.assert_allclose(best_accuracy, monitor.best_accuracy, atol=1e-8)

    metadata = {
        'model_name': name, 'run_id': RUN_ID, 'training_complete': True,
        'model_file': str(model_dir / 'best.keras'),
        'model_sha256': hashlib.sha256((model_dir / 'best.keras').read_bytes()).hexdigest(),
        'class_names': CLASS_NAMES, 'class_to_index': CLASS_TO_INDEX,
        'image_size': list(IMAGE_SIZE), 'color_mode': 'RGB',
        'resize': 'tf.image.resize bilinear; stretch to 224x224',
        'decode': 'tf.io.decode_image channels=3; no EXIF orientation transform',
        'preprocessing': preprocess_name, 'preprocessing_location': 'outside saved model',
        'tensorflow_version': tf.__version__,
        'keras_version': getattr(tf.keras, '__version__', None),
        'manifest_hashes': manifest_hashes, 'class_weights': None,
        'focal_alpha': FOCAL_ALPHA, 'focal_gamma': 2.0,
        'selection_metric': 'validation_macro_f1',
        'validation_macro_f1': monitor.best,
        'validation_accuracy': monitor.best_accuracy,
        'best_phase': monitor.best_phase, 'best_epoch': monitor.best_epoch,
        'training_seconds': round(time.time() - started, 2),
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'experiment': 'experiment_06_focal',
        'experiment_settings': EXPERIMENT_SETTINGS,
        'original_test_results_already_inspected': True,
        'test_images_used_for_gradient_updates': False,
        'checkpoint_selection_uses_validation_only': True,
    }
    write_json(metadata_path, metadata)
    history = pd.DataFrame(monitor.rows)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history['loss'], label='train')
    axes[0].plot(history['val_loss'], label='validation')
    axes[0].set_title('Loss')
    axes[1].plot(history['accuracy'], label='train')
    axes[1].plot(history['val_accuracy'], label='validation')
    axes[1].set_title('Accuracy')
    axes[2].plot(history['val_macro_f1'], label='validation macro F1')
    axes[2].set_title('Validation macro F1')
    for axis in axes:
        axis.set_xlabel('Recorded epoch (head then fine-tuning)')
        axis.legend()
    fig.tight_layout()
    fig.savefig(report_dir / 'training_curves.png', dpi=160)
    plt.close(fig)
    del loaded, model, base, monitor, train_ds, val_ds
    tf.keras.backend.clear_session()
    gc.collect()
    print(name, 'COMPLETE. Best validation macro F1:', metadata['validation_macro_f1'])
    return metadata


## Train one ResNet50 configuration


In [ ]:
experiment6_result = train_one_model('ResNet50')
display(pd.DataFrame([experiment6_result])[[
    'model_name', 'validation_macro_f1', 'validation_accuracy',
    'best_phase', 'best_epoch', 'training_seconds',
]])
print('Run ID:', RUN_ID)
print('Model folder:', RUN_MODELS / 'ResNet50')
print('Reports:', RUN_REPORTS / 'ResNet50')


## Compare checkpoints and export candidate


In [ ]:
# Compare saved validation results only; test predictions are never read.
run_ids = ['77751764c605', 'exp02_65b8c20800f3',
           'exp03_cbebb5733f81', 'exp05_e4604ccc3eab', RUN_ID]
rows = []
records = []
for run_id in run_ids:
    for model_name in ['MobileNetV2', 'EfficientNetB0', 'ResNet50']:
        path = MODEL_ROOT / run_id / model_name / 'metadata.json'
        if not path.is_file():
            if run_id in {RUN_ID, 'exp05_e4604ccc3eab'} and model_name != 'ResNet50':
                continue
            raise FileNotFoundError(f'Missing expected experiment metadata: {path}')
        info = json.loads(path.read_text())
        if not info.get('training_complete') or info['manifest_hashes'] != manifest_hashes:
            raise ValueError(f'Incomplete or incompatible experiment: {path}')
        per_class_path = REPORT_ROOT / run_id / model_name / 'validation_per_class_metrics.csv'
        scc = {}
        if per_class_path.is_file():
            table = pd.read_csv(per_class_path)
            scc = table.loc[table['class_label'] == 'SCC'].iloc[0].to_dict()
        row = {
            'run_id': run_id, 'model_name': model_name,
            'image_size': str(info['image_size']),
            'validation_accuracy': info['validation_accuracy'],
            'validation_macro_f1': info['validation_macro_f1'],
            'SCC_precision': scc.get('precision', np.nan),
            'SCC_recall': scc.get('recall', np.nan),
            'SCC_F1': scc.get('f1-score', np.nan),
        }
        rows.append(row)
        records.append(info)

comparison = pd.DataFrame(rows).sort_values(
    ['validation_macro_f1', 'run_id', 'model_name'],
    ascending=[False, True, True],
).reset_index(drop=True)
comparison.to_csv(RUN_REPORTS / 'all_experiments_validation_comparison.csv', index=False)
display(comparison.round(4))
winner_row = comparison.iloc[0]
winner = next(r for r in records if r['run_id'] == winner_row['run_id']
              and r['model_name'] == winner_row['model_name'])
print('Highest validation macro F1:', winner['run_id'], winner['model_name'])
print('Validation macro F1:', winner['validation_macro_f1'])
print('This is not a new test score.')

# Export to a NEW comparison package. Never overwrite earlier selected packages.
source = MODEL_ROOT / winner['run_id'] / winner['model_name'] / 'best.keras'
if hashlib.sha256(source.read_bytes()).hexdigest() != winner['model_sha256']:
    raise ValueError('Selected checkpoint checksum failed.')
package = PROJECT_ROOT / 'Models' / 'Selected' / ('final_review_' + RUN_ID)
package.mkdir(parents=True, exist_ok=True)
target = package / 'model.keras'
if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != winner['model_sha256']:
    raise ValueError('Different model already exists in this package; inspect it before proceeding.')
temporary = package / 'model.keras.tmp'
shutil.copy2(source, temporary)
if hashlib.sha256(temporary.read_bytes()).hexdigest() != winner['model_sha256']:
    raise ValueError('Copied model checksum failed.')
temporary.replace(target)
package_info = {
    **winner,
    'model_file': str(target),
    'comparison_run_id': RUN_ID,
    'selection_basis': 'highest validation macro F1 among recorded trained checkpoints',
    'original_test_results_already_inspected': True,
    'checkpoint_selection_uses_validation_only': True,
    'new_test_evaluation_performed': False,
}
write_json(package / 'metadata.json', package_info)
write_json(package / 'class_names.json', CLASS_NAMES)
write_json(RUN_REPORTS / 'final_candidate.json', package_info)
print('Verified candidate package:', package)
print('Candidate source run:', winner['run_id'])
print('Candidate image size:', winner['image_size'])
print('Training comparison complete. No test evaluation was performed.')
print('Integration remains paused. Performance requirements are not declared met.')


## Follow-up evaluation — enable after fixing the candidate

The cell below uses the selected metadata's input size, model hash and source run. Set `RUN_FOLLOWUP_EVALUATION = True` only after reviewing the comparison above. Keep the candidate fixed regardless of test rankings. A completed evaluation is loaded on rerun rather than scored again. Read accuracy, macro F1 **and each class's sensitivity**, including the existing README's 0.80 sensitivity target.

The report records darker-skin performance as unestablished; changing a Boolean does not supply evidence. Any future external dataset needs permitted use, diagnosis compatibility and independent patients before it can address this gap.


In [ ]:
# Run once after reviewing the fixed validation-selected candidate above.
# Set True to perform the final FOLLOW-UP evaluation on the already inspected test set.
RUN_FOLLOWUP_EVALUATION = False

from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_recall_fscore_support,
    f1_score, roc_auc_score, roc_curve,
)

def compute_metrics(labels, probabilities):
    predicted = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predicted, labels=np.arange(3))
    tp = np.diag(matrix).astype(float)
    fn = matrix.sum(axis=1) - tp
    fp = matrix.sum(axis=0) - tp
    tn = matrix.sum() - tp - fn - fp
    specificity = np.divide(tn, tn + fp, out=np.full(3, np.nan), where=(tn + fp) != 0)
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, predicted, labels=np.arange(3), zero_division=0,
    )
    auc_values = [
        float(roc_auc_score((labels == index).astype(int), probabilities[:, index]))
        if np.unique(labels == index).size == 2 else np.nan
        for index in range(3)
    ]
    per_class = pd.DataFrame({
        'class_label': CLASS_NAMES, 'support': support,
        'precision': precision, 'sensitivity_recall': recall,
        'specificity_ovr': specificity, 'f1_score': f1,
        'roc_auc_ovr': auc_values,
        'true_positive': tp.astype(int), 'false_negative': fn.astype(int),
        'false_positive': fp.astype(int), 'true_negative': tn.astype(int),
    })
    summary = {
        'accuracy': float(accuracy_score(labels, predicted)),
        'macro_precision': float(precision.mean()),
        'macro_sensitivity': float(recall.mean()),
        'macro_specificity': float(np.nanmean(specificity)),
        'macro_f1': float(f1.mean()),
        'macro_roc_auc_ovr': float(roc_auc_score(
            labels, probabilities, labels=np.arange(3), multi_class='ovr', average='macro',
        )),
        'test_records': len(labels),
    }
    return summary, per_class, matrix

def save_figures(name, labels, probabilities, matrix, folder):
    fig, ax = plt.subplots(figsize=(6, 5))
    display_image = ax.imshow(matrix, cmap='Blues')
    fig.colorbar(display_image, ax=ax)
    ax.set(xticks=np.arange(3), yticks=np.arange(3), xticklabels=CLASS_NAMES,
           yticklabels=CLASS_NAMES, xlabel='Predicted class', ylabel='Actual class',
           title=name + ' — test confusion matrix')
    for row in range(3):
        for column in range(3):
            ax.text(column, row, str(matrix[row, column]), ha='center', va='center',
                    color='white' if matrix[row, column] > matrix.max() / 2 else 'black')
    fig.tight_layout()
    fig.savefig(folder / 'confusion_matrix.png', dpi=180)
    plt.show()
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(6, 5))
    for index, class_name in enumerate(CLASS_NAMES):
        binary = (labels == index).astype(int)
        fpr, tpr, _ = roc_curve(binary, probabilities[:, index])
        auc = roc_auc_score(binary, probabilities[:, index])
        ax.plot(fpr, tpr, label=f'{class_name}: AUC={auc:.3f}')
    ax.plot([0, 1], [0, 1], '--', color='grey')
    ax.set(xlabel='False positive rate', ylabel='True positive rate', title=name + ' — one-vs-rest ROC')
    ax.legend()
    fig.tight_layout()
    fig.savefig(folder / 'roc_curves.png', dpi=180)
    plt.show()
    plt.close(fig)


def evaluate_fixed_candidate():
    selected = json.loads((package / 'metadata.json').read_text())
    candidate_path = package / 'model.keras'
    if hashlib.sha256(candidate_path.read_bytes()).hexdigest() != selected['model_sha256']:
        raise ValueError('Candidate model checksum differs.')
    if selected['class_names'] != CLASS_NAMES:
        raise ValueError('Candidate class order differs.')
    for split in ['train', 'validation', 'test']:
        path = SPLIT_ROOT / f'{split}_metadata.csv'
        if hashlib.sha256(path.read_bytes()).hexdigest() != selected['manifest_hashes'][split]:
            raise ValueError('Split manifest changed after candidate selection.')

    output = PROJECT_ROOT / 'Reports' / 'Evaluation' / package.name
    output.mkdir(parents=True, exist_ok=True)
    marker = output / 'evaluation_manifest.json'
    if marker.exists():
        saved = json.loads(marker.read_text())
        if saved['model_sha256'] != selected['model_sha256'] or saved['manifest_hashes'] != selected['manifest_hashes']:
            raise ValueError('A different evaluation exists in this folder; do not overwrite it.')
        print('This candidate already has completed follow-up results; loading them without rescoring.')
        display(pd.read_csv(output / 'per_class_metrics.csv'))
        display(pd.read_csv(output / 'requirements_check.csv'))
        print('Reports:', output)
        return saved

    size = tuple(selected['image_size'])
    preprocessors = {
        'resnet50': tf.keras.applications.resnet50.preprocess_input,
        'mobilenet_v2': tf.keras.applications.mobilenet_v2.preprocess_input,
        'efficientnet_b0': tf.keras.applications.efficientnet.preprocess_input,
    }
    if selected['preprocessing'] not in preprocessors or selected.get('preprocessing_location') != 'outside saved model':
        raise ValueError('Unknown candidate preprocessing contract.')
    preprocess = preprocessors[selected['preprocessing']]
    def read_candidate_image(path):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        return tf.image.resize(tf.cast(image, tf.float32), size, method='bilinear')

    tf.keras.backend.clear_session()
    gc.collect()
    model = tf.keras.models.load_model(candidate_path, compile=False)
    source_reports = PROJECT_ROOT / 'Reports' / 'Training' / selected['run_id'] / selected['model_name']
    reference_frame = pd.read_csv(source_reports / 'validation_reference_images.csv')
    with np.load(source_reports / 'validation_reference.npz') as archive:
        reference_scores = archive['scores'].copy()
        reference_labels = archive['labels'].copy()
    if not np.array_equal(reference_labels, reference_frame['label_index'].to_numpy()):
        raise ValueError('Saved validation reference identifiers differ.')
    reference_images = tf.stack([read_candidate_image(str(path)) for path in reference_frame['image_path']])
    np.testing.assert_allclose(
        model.predict(preprocess(reference_images), verbose=0),
        reference_scores, rtol=1e-3, atol=1e-4,
    )
    test = frames['test'].copy().reset_index(drop=True)
    labels = test['label_index'].to_numpy(dtype=np.int32)
    for path in test['image_path']:
        if not Path(path).is_file():
            raise FileNotFoundError(path)
    test_ds = tf.data.Dataset.from_tensor_slices(test['image_path'].astype(str).to_numpy())
    test_ds = test_ds.map(read_candidate_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE)
    test_ds = test_ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
    probabilities = model.predict(test_ds, verbose=1)
    if probabilities.shape != (len(test), 3) or not np.isfinite(probabilities).all() or (probabilities < 0).any():
        raise ValueError('Invalid model scores.')
    np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, atol=1e-5)
    summary, per_class, matrix = compute_metrics(labels, probabilities)
    summary.update({
        'model_name': selected['model_name'], 'source_run_id': selected['run_id'],
        'image_size': list(size), 'model_sha256': selected['model_sha256'],
        'evaluation_type': 'follow-up on previously inspected test set',
    })
    per_class.to_csv(output / 'per_class_metrics.csv', index=False)
    pd.DataFrame(matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(output / 'confusion_matrix.csv')
    predictions = test.copy()
    predictions['predicted_class'] = np.asarray(CLASS_NAMES)[probabilities.argmax(axis=1)]
    for index, name in enumerate(CLASS_NAMES):
        predictions[f'score_{name}'] = probabilities[:, index]
    predictions.to_csv(output / 'test_predictions.csv', index=False)
    predictions.loc[predictions['class_label'] != predictions['predicted_class']].to_csv(output / 'misclassified_images.csv', index=False)
    save_figures(selected['model_name'], labels, probabilities, matrix, output)
    requirements = [
        {'requirement': 'Test accuracy >= 0.85', 'observed': summary['accuracy'], 'met': summary['accuracy'] >= 0.85},
        {'requirement': 'Test macro F1 > 0.80', 'observed': summary['macro_f1'], 'met': summary['macro_f1'] > 0.80},
    ]
    for row in per_class.to_dict('records'):
        requirements.append({'requirement': row['class_label'] + ' sensitivity >= 0.80',
                             'observed': row['sensitivity_recall'], 'met': row['sensitivity_recall'] >= 0.80})
    requirement_table = pd.DataFrame(requirements)
    requirement_table.to_csv(output / 'requirements_check.csv', index=False)
    summary['measured_performance_targets_met'] = bool(requirement_table['met'].all())
    summary['darker_skin_performance_established'] = False
    summary['skin_tone_note'] = 'Existing IV-VI validation support is only 3 BCC and 1 SCC images, with no Melanoma; independent representative evaluation remains needed.'
    write_json(output / 'test_metrics.json', summary)
    write_json(package / 'test_metrics.json', summary)
    per_class.to_csv(package / 'per_class_metrics.csv', index=False)
    evaluation = {
        **summary, 'manifest_hashes': selected['manifest_hashes'],
        'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
        'selection_basis': 'validation macro F1; candidate fixed before this evaluation',
        'tensorflow_version': tf.__version__,
    }
    # Completion marker written last; reruns then display rather than rescore.
    write_json(marker, evaluation)
    display(per_class.round(4))
    display(requirement_table)
    print('Follow-up test accuracy:', summary['accuracy'])
    print('Follow-up test macro F1:', summary['macro_f1'])
    print('Measured performance targets met:', summary['measured_performance_targets_met'])
    print('Darker-skin performance established: False — inadequate real subgroup evidence.')
    print('Reports:', output)
    print('Candidate:', package)
    del model, test_ds, reference_images
    tf.keras.backend.clear_session()
    gc.collect()
    return evaluation

if RUN_FOLLOWUP_EVALUATION:
    final_evaluation = evaluate_fixed_candidate()
else:
    print('Evaluation paused. Review the fixed candidate above, then set RUN_FOLLOWUP_EVALUATION = True to evaluate once.')


## Save and report

Save this notebook to Drive. Save a copy to `Chetalam/cnn-skin-cancer-classification`, branch `main` after merging the reviewed PR, path `notebooks/07_final_training_and_evaluation.ipynb`. Commit description: `Record final focal-loss comparison and follow-up evaluation`.

Report the actual selected run, input size, validation macro F1, test accuracy/macro F1, per-class sensitivity and SCC precision/recall/F1. Preserve experiments that did not improve. Do not claim that skin-tone fairness, Kenyan population performance or clinical validity has been established.

Checkpoint and report files are saved to Drive automatically. On disconnect rerun in order: completed training is skipped after checksum verification, while incomplete training is archived and restarted. Do not change the original split manifests or upload patient data to GitHub.
